In [ ]:
import json

from rockyclickup.wrapper import Session
from rockyclickup.models import DataFile
from rockyclickup.utils import response_to_model
from rockyclickup.models_base import Task

In [ ]:
rcu = Session()

In [ ]:
has_updated = False
prior = None

In [ ]:
task_json = rcu.get_task_by_id("868jqnwpj")
date_updated = task_json.get("date_updated")

print(f"{'updated' if date_updated != prior else 'not updated'}")

prior = date_updated

In [ ]:
# task name
# comment
# status change
# priority change
# type change
# assignee changed (remove or add)
# field change
# tag added

In [ ]:
task_model = response_to_model(task_json)

In [ ]:
task_json

In [66]:
import datetime as dt

class DateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (dt.date, dt.datetime)):
            try:
                return str(int(obj.timestamp()))

            except:
                return None
        
        return super().default(obj)

In [ ]:
# need to check if we need to update the datafile in the database

# i think we should check two things:
    # 1) check if the date_updated matches what the database says
        # NOTE we should make sure we know what can cause the clickup field "date_updated" to be changed
            # KNOWN EVENTS
                # comments being added
                # status changes

        # if mismatch: just update everything


    # 2) checksum
        # get a narrowed dict of the clickup task 
        # get hash and compare against recorded hash in db


# columns required to do this:
    # checkum
    # date_task_updated             # this is what clickup gives us, not when the database row was modified
    # fields we want to hash to compare
        # task name
        # file name
        # directory
        # assignees
        # file_category
        # status
        # archived

# if an update is required, we must update not just the row values in the database but the raw_json value as well


In [69]:
narrowed_dict = {
    m: getattr(task_model, k)
    for k, m in {
        "id":                    "task_id",
        "name":                  "task_name",
        "status":                "status",
        "archived":              "archived",
        "assignees":             "assignees",
        "ftp_filename":          "file_name",
        "ftp_directory":         "file_directory",
        "file_category":         "category",
        "file_date":             "file_date",
        "received":              "date_received",
        "date_created":          "date_task_created",
        "date_updated":          "date_task_updated",
        "creator":               "task_author", # this isnt getting captured in the `response to model` function   
        "inbox":                 "datacards",
    }.items()
}

print(json.dumps(narrowed_dict, indent=4, cls=DateTimeEncoder))

{
    "task_id": "868jqnwpj",
    "task_name": "this is a term.webp",
    "status": "new",
    "archived": false,
    "assignees": null,
    "file_name": "this is a term.webp",
    "file_directory": "/Clients/Snow Globe Theater-RMRSNOW/File Feeds",
    "category": "COBRA",
    "file_date": null,
    "date_received": "1779321600",
    "date_task_created": "1779399300",
    "date_task_updated": "1779466260",
    "task_author": null,
    "datacards": [
        "8688frrac"
    ]
}


In [77]:
from hashlib import sha256


def get_hash(dictionary):
    encoded = json.dumps(dictionary, sort_keys=True, cls=DateTimeEncoder).encode("utf-8")

    return sha256(encoded).hexdigest()

my_hash = get_hash(narrowed_dict)

my_hash

'5c036648c5a1160a29061d86fb35ba173c38e9b2b998d79cac800c305ba43514'

In [78]:
len(my_hash)

64